# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from itertools import product
import geopandas as gpd
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm

root_path = Path(globals()['_dh'][0]).resolve().parent
sys.path.append(str(root_path))

from paths import config_path, input_path, api_path
from library.utilities import get_path

In [2]:
# 2. Load the configuration

with (config_path / 'prototype.json').open('r', encoding='utf-8') as file:
    config = json.load(file)

In [3]:
# 3. Load the base demand

base_demand = pd.read_csv(input_path / config['loader']['properties']['access'] / 'base_demand' / get_path(config['loader']['name'],config['loader']['properties'], "csv"), index_col=['timestamp'], parse_dates=['timestamp'])

In [4]:
# 4 Execute transformations - Sort the transformations array

transformers = config['transformers']
transformers.sort(key=lambda x: x["order"])

In [5]:
# 4.1. Execute transformations - Split base demand (national) into municipalities

input = {input['name']: pd.read_csv(input_path / transformers[0]['access'] / input['path'], usecols=input['columns'], dtype={input['index']: str}, index_col=[input['index']]) for input in transformers[0]['inputs']}

municipality_index = pd.MultiIndex.from_product(
    [input['municipality_energy_split'].index, base_demand.index],
    names=['municipality', 'timestamp']
)

municipal_demand = pd.DataFrame(
    (input['municipality_energy_split']['ratio'].values[:, None] * base_demand['Sweden'].values).ravel()[:, None], 
    index=municipality_index, 
    columns=['demand']
)

In [6]:
# 4.2. Execute transformations - Apply growth over time

# Define growth parameters
target_energy = 330  # TWh projected for 2045
yearly_growth = (target_energy / (base_demand['Sweden'].sum() / 1_000_000)) ** (1 / 20) - 1  # About 4.32% growth yearly
twenty_yrs_in_hours = (datetime(2045, 1, 1, 0, 0, 0) - datetime(2025, 1, 1, 0, 0, 0)).total_seconds() / 3600
hourly_growth = (target_energy / (base_demand['Sweden'].sum() / 1_000_000)) ** (1 / twenty_yrs_in_hours) - 1

# Generate the new timestamps for the extended demand
start_time = pd.Timestamp("2025-01-01 00:00:00")
end_time = pd.Timestamp("2045-12-31 23:00:00")
new_timestamps = pd.date_range(start=start_time, end=end_time, freq='h')

# Calculate the growth factors for all hours
hours_elapsed = np.arange(len(new_timestamps))  # Elapsed hours as a 1D array
growth_factors = (1 + hourly_growth) ** hours_elapsed  # Shape: (len(new_timestamps),)

# Prepare the new DataFrame structure
municipalities = municipal_demand.index.get_level_values('municipality').unique()
new_index = pd.MultiIndex.from_product(
    [municipalities, new_timestamps], names=["municipality", "timestamp"]
)

# Extract hourly demand patterns for 2024
hourly_demand_2024 = (
    municipal_demand.loc[
        municipal_demand.index.get_level_values('timestamp').year == 2024
    ]
    .reset_index()
)

# Repeat the 2024 demand pattern to match the new timestamps
hourly_demand_pattern = hourly_demand_2024.groupby("municipality")["demand"].apply(
    lambda x: np.tile(x.values, len(new_timestamps) // len(x) + 1)[:len(new_timestamps)]
)

# Convert hourly demand pattern back to a 2D array (municipalities x timestamps)
base_demand_repeated = np.vstack(hourly_demand_pattern.values)

# Apply growth factors
extended_demand_values = base_demand_repeated * growth_factors  # Shape: (num_municipalities, len(new_timestamps))

# Create the extended demand DataFrame
extended_municipal_demand = pd.DataFrame(
    data=extended_demand_values.flatten(),
    index=new_index,
    columns=["demand"]
)

In [7]:
# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']

## TODO: Make this less naive

## This transform is not realistic. It does the following:
##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality
##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder 

# Extract July data for each year
july_data = extended_municipal_demand.loc[
    extended_municipal_demand.index.get_level_values('timestamp').month == 7
]

# Compute the lowest July value per municipality and year
lowest_july_per_year = (
    july_data
    .groupby([july_data.index.get_level_values('municipality'),
              july_data.index.get_level_values('timestamp').year])['demand']
    .min()
)

# Convert to DataFrame and calculate industry demand
lowest_july_per_year = lowest_july_per_year.to_frame(name='lowest_july')
lowest_july_per_year['industry_demand'] = lowest_july_per_year['lowest_july'] * 0.5

# Create extended_municipal_sector_demand and merge industry demand back with the full extended demand
# Add a column to join by year
extended_municipal_sector_demand = extended_municipal_demand.copy()
extended_municipal_sector_demand['year'] = extended_municipal_sector_demand.index.get_level_values('timestamp').year

# Join the calculated industry demand
extended_municipal_sector_demand = extended_municipal_sector_demand.join(
    lowest_july_per_year['industry_demand'], 
    on=['municipality', 'year']
)

# Industry demand is constant across each municipality and year
extended_municipal_sector_demand['industry'] = extended_municipal_sector_demand['industry_demand']

# Compute the remainder for buildings and transport
extended_municipal_sector_demand['remainder'] = extended_municipal_sector_demand['demand'] - extended_municipal_sector_demand['industry']

# Split remainder into buildings (95%) and transport (5%)
extended_municipal_sector_demand['buildings'] = extended_municipal_sector_demand['remainder'] * 0.95
extended_municipal_sector_demand['transport'] = extended_municipal_sector_demand['remainder'] * 0.05

# Drop unnecessary intermediate columns
extended_municipal_sector_demand = extended_municipal_sector_demand.drop(columns=['industry_demand', 'remainder', 'year'])

# Rename 'demand' to 'total' for clarity
extended_municipal_sector_demand = extended_municipal_sector_demand.rename(columns={'demand': 'total'})


In [8]:
# 5. Write output

# Write yearly per municipality (1h, 3h, 1d, 1w, 1m, 1y), for the country as a whole

# TODO: This script currently takes an hour+ to run through 20 years and 290 municipalities and 5 resolutions. I need to improve this.

geos = extended_municipal_sector_demand.index.get_level_values('municipality').unique()
years = extended_municipal_sector_demand.index.get_level_values('timestamp').year.unique()

aggregations = [
    {"resolution": "1h", "aggregation": ["none"]},
    {"resolution": "3h", "aggregation": ["mean", "sum", "min", "max"]},
    {"resolution": "1d", "aggregation": ["mean", "sum", "min", "max"]},
    {"resolution": "1W", "aggregation": ["mean", "sum", "min", "max"]},
    {"resolution": "1ME", "aggregation": ["mean", "sum", "min", "max"]},
    {"resolution": "1YE", "aggregation": ["sum"]}
]

melted_aggregations = [
    {"resolution": item["resolution"], "aggregation": stat}
    for item in aggregations
    for stat in item["aggregation"]
]

def aggregate(df, res, stat):
    if stat == 'sum':
        return df.resample(res).sum()
    elif stat == 'mean':
        return df.resample(res).mean()
    elif stat == 'max':
        return df.resample(res).max()
    elif stat == 'min':
        return df.resample(res).min()
    else:
        return df
    
def header(stat):
    if stat == 'sum':
        return 'total demand (MWh)'
    elif stat == 'mean':
        return 'average demand (MW)'
    elif stat == 'max':
        return 'max demand (MW)'
    elif stat == 'min':
        return 'min demand (MW)'
    else:
        return 'average demand (MW)'

In [9]:
# Write demand summed over geos

print("Writing demand summed over geos")
country_data = extended_municipal_sector_demand.groupby(level='timestamp').sum()
for year in tqdm(years, desc="Progress (years)", unit="years"):
    yearly_data = country_data.loc[country_data.index.year == year]
    for agg in melted_aggregations:
        aggregate(yearly_data, agg['resolution'], agg['aggregation']).to_csv(
            api_path / f"demand_t,geography=00,resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year}.csv.gz",
            compression='gzip'
        )

Writing demand summed over geos


Progress (years): 100%|██████████| 21/21 [00:13<00:00,  1.54years/s]


In [10]:
# Write demand per geo and year

def process_geo_year(task):
    geo, year = task  # Unpack the task tuple
    municipal_data = extended_municipal_sector_demand.xs(geo, level='municipality')
    yearly_data = municipal_data.loc[municipal_data.index.year == year]
    
    for agg in melted_aggregations:
        aggregate(yearly_data, agg['resolution'], agg['aggregation']).to_csv(
            api_path / f"demand_t,geography={geo},resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year}.csv.gz",
            compression='gzip'
        )

# Create tasks as (geo, year) pairs
tasks = list(product(geos, years))

# Use ProcessPoolExecutor with a top-level function
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(process_geo_year, tasks), total=len(tasks), desc="Geo-Year Pairs"))


Geo-Year Pairs: 100%|██████████| 6090/6090 [21:36<00:00,  4.70it/s]  


In [11]:
# Write the geojson files

geographies_gdp = gpd.read_file(input_path / 'public' / 'geographies' / 'georef-sweden-kommun@public.geojson', encoding='utf-8')

municipalities = geographies_gdp[['year', 'kom_code', 'kom_name', 'geometry']].copy()

# Group data by year and municipality once for all aggregations
grouped = extended_municipal_sector_demand.groupby([extended_municipal_sector_demand.index.get_level_values('timestamp').year, 'municipality'])

yearly_stats = {
    'sum': grouped.sum(),
    'mean': grouped.mean(),
    'max': grouped.max(),
    'min': grouped.min(),
}

units = {
    'sum': 'MWh',
    'mean': 'MW',
    'max': 'MW',
    'min': 'MW'
}

print("Writing geojson files")

# Process data for each year
for year in tqdm(years, desc="Progress (years)", unit="years"):
    municipalities['year'] = year

    # Add statistics and sectors to the GeoDataFrame
    for stat, data in yearly_stats.items():
        year_data = data.loc[year]  # Filter data for the specific year
        
        for sector in ['total', 'industry', 'buildings', 'transport']:
            col_name = f"{stat}_{sector}"
            municipalities[col_name] = municipalities['kom_code'].map(year_data[sector])
            municipalities[f"{col_name}_unit"] = units[stat]
    
    # Save the GeoJSON file
    output_file = api_path / f"demand_geo,resolution=1YE,sector=all,aggregation=all,year={year}.geojson"
    municipalities.to_file(output_file, driver="GeoJSON", encoding="utf-8")


Writing geojson files


Progress (years): 100%|██████████| 21/21 [00:16<00:00,  1.30years/s]


In [13]:
# Write the parameters.json

print("Writing parameters.json")

geographies = geographies_gdp[['kom_type', 'kom_code', 'kom_name', 'lan_code', 'lan_name']].copy()

geographies = geographies.rename(columns={
    "kom_type": "type",
    "kom_code": "id",
    "kom_name": "name",
    "lan_code": "parent_id",
    "lan_name": "parent_name"
})

## Add Sweden as a whole
new_row = pd.DataFrame({
    "type": ["Land"],
    "id": ["00"],
    "name": ["Sverige"],
    "parent_name": [""],
    "parent_id": [""]
})

geographies = pd.concat([geographies, new_row], ignore_index=True)

parameters = {
    'years': list(range(config['start-year'], config['end-year'])),
    'geographies': geographies.to_dict(orient="records"),
    'aggregations': config['output']['properties']['aggregations'],
    'sectors': config['output']['properties']['sectors']
}

(api_path / "parameters.json").write_text(json.dumps(parameters, indent=4, ensure_ascii=False), encoding='utf-8')


Writing parameters.json


54933